# M2_E4: multi-pass blocking + LightGBM

This notebook orchestrates the reusable `experiments.M2_E4.run_final_inference` implementation. It fits LightGBM using training records and ground truth only, then builds the candidate index from test Sources 2 and 3 and generates predictions for every test Source 1 entity. Training and test candidate indexes are disk-backed SQLite databases, deleted after each phase. The training sample and negative candidate sampling are deterministic; the prediction threshold is fixed at 0.5 and is not selected using test data.

Running all cells writes the actual test submission files under `output/`: `matching_results.tsv` and `candidate_pairs.tsv`. `output/metrics.json` contains run metadata and is not uploaded as a submission TSV. A full run processes the complete train and test data and may take substantial time and temporary disk space.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "preprocessing.py").is_file()
     and (path / "student_resource" / "dataset" / "train").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open/run this notebook from within the ML-Challenge project.")
sys.path.insert(0, str(PROJECT_ROOT))
from experiments.M2_E4.run_experiment import ExperimentConfig, run_final_inference
config = ExperimentConfig(
    train_dir=PROJECT_ROOT / "student_resource" / "dataset" / "train",
    output_dir=PROJECT_ROOT / "experiments" / "M2_E4",
    random_seed=42,
    max_train_entities=25_000,
    negative_samples_per_entity=30,
    chunk_size=20_000,
    max_bucket_size=100,
    prediction_threshold=0.5,
)
metrics = run_final_inference(
    config,
    test_dir=PROJECT_ROOT / "student_resource" / "dataset" / "test",
    submission_dir=PROJECT_ROOT / "output",
)
{key: metrics[key] for key in (
    "source1_count", "target_count", "prediction_count",
    "candidate_pair_count", "candidate_count_average",
    "candidate_count_median", "candidate_count_p95", "candidate_count_max",
    "candidate_reduction_ratio", "runtime_seconds",
)}

After the notebook finishes, validate the actual test files from the `student_resource/` directory:

```powershell
python utils/validate_submission.py --matching ../output/matching_results.tsv --candidate ../output/candidate_pairs.tsv --test-dir dataset/test
```

Submit only the two validated TSVs in the format required by the challenge; `metrics.json` is a run report. Never replace or write files inside `student_resource/dataset/`.